# 02 — Trend Signal Construction

This notebook converts monthly ETF prices into cross-asset trend signals for the portfolio backtest. The signal is timestamped at the month-end when it becomes observable; the trading lag is applied in the backtest notebook.


In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.graph_objects as go

pd.options.display.float_format = "{:,.4f}".format

DATA_DIR = Path("data")
SIGNAL_DIR = Path("results") / "trend_signal"
SIGNAL_DIR.mkdir(parents=True, exist_ok=True)

prices = pd.read_csv(DATA_DIR / "prices.csv", index_col=0, parse_dates=True)
returns = pd.read_csv(DATA_DIR / "returns.csv", index_col=0, parse_dates=True)

default_layout = dict(
    template="plotly_white",
    height=500,
    margin=dict(t=70, l=60, r=40, b=60),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)


In [ ]:
def compute_signal_summary(signal, raw_trend):
    """Summarize signal strength, switching activity, and underlying trend behavior."""
    common_index = signal.index.intersection(raw_trend.index)
    signal_block = signal.loc[common_index]
    trend_block = raw_trend.loc[common_index]

    valid_change = signal_block.notna() & signal_block.shift(1).notna()
    signal_changes = (signal_block.diff().ne(0) & valid_change).sum()

    summary = pd.DataFrame({
        "Average Signal (%)": signal_block.mean() * 100,
        "Signal Changes": signal_changes,
        "Average Trend Return (%)": trend_block.mean() * 100,
        "Trend Return Volatility (%)": trend_block.std() * 100,
    })

    return summary.sort_values("Average Signal (%)", ascending=False).round(2)


## 1. Monthly returns and 12-month trend signal

Daily adjusted prices are converted to month-end prices and monthly returns. If the latest download includes an incomplete current month, that row is dropped so that a partial month is not treated as a full monthly observation.

The first signal is a 12-month absolute trend rule: an asset receives a score of 1 when its trailing 12-month price return is positive and 0 when it is negative. Missing lookback periods are left as missing rather than being treated as negative signals.


In [ ]:
monthly_prices = prices.resample("ME").last()

current_month = pd.Timestamp.today().to_period("M")
if monthly_prices.index[-1].to_period("M") == current_month:
    monthly_prices = monthly_prices.iloc[:-1]

monthly_returns = monthly_prices.pct_change().dropna(how="any")

print("Monthly price data shape:", monthly_prices.shape)
display(monthly_prices.tail())


In [ ]:
lookback_months = 12

trend_12m = monthly_prices.pct_change(lookback_months)
signal_12m = trend_12m.gt(0).astype(float).where(trend_12m.notna())

signal_12m.tail()


## 2. 12-month trend diagnostics

The diagnostic charts focus on a representative subset of assets to keep the output readable while still covering equities, duration, gold, and commodities. The in-sample window ends in 2017; later notebooks use the post-2017 period for portfolio evaluation.


In [ ]:
IS_END = "2017-12-31"
plot_start = "2007-01-01"
plot_end = IS_END

assets_to_plot = ["SPY", "EFA", "IEF", "TLT", "GLD", "DBC"]
trend_12m_plot = trend_12m.loc[plot_start:plot_end, assets_to_plot]
signal_12m_plot = signal_12m.loc[plot_start:plot_end, assets_to_plot]


In [ ]:
fig = go.Figure()
for asset in assets_to_plot:
    fig.add_trace(go.Scatter(x=trend_12m_plot.index, y=trend_12m_plot[asset], mode="lines", name=asset))

fig.add_hline(y=0, line_dash="dash")
fig.update_layout(
    **default_layout,
    title="12-Month Trend Returns",
    xaxis_title="Date",
    yaxis_title="Trailing 12-Month Return",
)
fig.update_yaxes(tickformat=".0%")
fig.show()


The trend-return chart is a regime check rather than a performance result. The rule should switch off during sustained weakness in risk assets and behave differently across defensive and inflation-sensitive assets.


In [ ]:
fig = go.Figure()
for asset in assets_to_plot:
    fig.add_trace(
        go.Scatter(
            x=signal_12m_plot.index,
            y=signal_12m_plot[asset],
            mode="lines",
            name=asset,
            line_shape="hv",
        )
    )

fig.update_layout(
    **default_layout,
    title="12-Month Trend Signals",
    xaxis_title="Date",
    yaxis_title="Signal: 1 = positive trend, 0 = negative trend",
)
fig.show()


### In-sample and full-sample signal summaries

The average signal measures how often, or how strongly, an asset is eligible for risk allocation. Signal changes indicate potential turnover pressure before portfolio weighting is applied.


In [ ]:
signal_summary_12m_is = compute_signal_summary(
    signal=signal_12m.loc[:IS_END],
    raw_trend=trend_12m.loc[:IS_END],
)

signal_summary_12m_is


In [ ]:
signal_summary_12m_full = compute_signal_summary(
    signal=signal_12m,
    raw_trend=trend_12m,
)

signal_summary_12m_full


## 3. Multi-horizon trend signal

A single 12-month rule is transparent but dependent on one lookback window. The final signal averages three binary trend rules based on 3-month, 6-month, and 12-month returns. This keeps the rule simple while reducing dependence on one horizon.

Signal interpretation: 0.00 = no trend support, 0.33 = weak support, 0.67 = moderate support, and 1.00 = broad support across all three horizons.


In [ ]:
trend_3m = monthly_prices.pct_change(3)
trend_6m = monthly_prices.pct_change(6)
trend_multi = (trend_3m + trend_6m + trend_12m) / 3

signal_3m = trend_3m.gt(0).astype(float).where(trend_3m.notna())
signal_6m = trend_6m.gt(0).astype(float).where(trend_6m.notna())
signal_multi = (signal_3m + signal_6m + signal_12m) / 3


In [ ]:
signal_multi_plot = signal_multi.loc[plot_start:plot_end, assets_to_plot]

fig = go.Figure()
for asset in assets_to_plot:
    fig.add_trace(
        go.Scatter(
            x=signal_multi_plot.index,
            y=signal_multi_plot[asset],
            mode="lines",
            name=asset,
            line_shape="hv",
        )
    )

fig.update_layout(
    **default_layout,
    title="Multi-Horizon Trend Signal",
    xaxis_title="Date",
    yaxis_title="Signal score",
)
fig.show()


In [ ]:
signal_summary_multi = compute_signal_summary(
    signal=signal_multi,
    raw_trend=trend_multi,
)

signal_summary_multi


## 4. Final signal file for portfolio construction

The multi-horizon signal is selected for the portfolio stage because it is simple, interpretable, and less dependent on a single lookback window. The backtest notebook will apply a one-month trading lag when converting these month-end signals into returns.


In [ ]:
common_index = monthly_returns.index.intersection(signal_multi.index)

final_signal = signal_multi.loc[common_index]
monthly_returns_aligned = monthly_returns.loc[common_index]

valid_dates = final_signal.dropna(how="any").index
final_signal = final_signal.loc[valid_dates]
monthly_returns_aligned = monthly_returns_aligned.loc[valid_dates]

print("Final signal shape:", final_signal.shape)
print("Aligned monthly returns shape:", monthly_returns_aligned.shape)
display(final_signal.tail())


In [ ]:
final_signal.to_csv(SIGNAL_DIR / "final_signal.csv")
monthly_returns_aligned.to_csv(SIGNAL_DIR / "monthly_returns_aligned.csv")

print("Saved trend-signal files to:", SIGNAL_DIR.resolve())
